# Error Analysis — LoRA fine-tuned DINOv3 ViT-S/16 Plus on DF40 `test_data_v3`

Aggregate metrics (balanced acc 0.902, val AUC 0.960) hide *where* the model fails. This notebook loads the
**LoRA fine-tuned** DINOv3 ViT-S/16 Plus classifier (`outputs/finetune/vit_lora_finetuned.pt`, 111 MB) and
dissects its False Negatives / False Positives on a **balanced** test set built from the DF40 manifest.

**Model:** ViT-S/16 Plus + LoRA · `embed_dim=384`, `depth=12`, `heads=6`, gated-MLP · LoRA `r=16, alpha=32` on `q_proj`,`v_proj`
(0.3M trainable of 29M) · checkpoint is self-contained (full backbone + LoRA + head; no separate weights file needed).
**Test set:** 1,177 real + 1,177 fake (stratified by method) from `test_data_v3/manifest.csv` — balanced by design.

> ⚠️ The model is tiny (29M) — prediction on 2,354 images takes well under a minute on GPU (predictions cached to
> `predictions.npz`). All analysis cells run on CPU.
> ⚠️ `data_train` real identities (`cdc:*`, `ffc:*`) overlap with the test set's real identities — the "real" class is
> partially *in-distribution*; treat absolute numbers as optimistic, the relative per-method/domain structure as informative.
> ⚠️ No `pandas` — this notebook uses only `csv` / `numpy` / `matplotlib` / `sklearn`.

In [11]:
# ============================================================
# Step 1: Setup — paths, device, constants, output dirs
# ============================================================
import os, sys, csv, json, random
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT).endswith("notebooks"):
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# --- Model: LoRA fine-tuned DINOv3 ViT-S/16 Plus ---
CHECKPOINT = PROJECT_ROOT / "outputs/finetune/vit_lora_finetuned.pt"
IMG_SIZE = 256
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BATCH = 64

# --- Data ---
MANIFEST  = Path("/workspace/data/test_data_v3/manifest.csv")
TEST_ROOT = Path("/workspace/data/test_data_v3")

# --- Outputs ---
OUT = PROJECT_ROOT / "experiments/results/error_analysis_lora"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42

EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CsvImageDataset(Dataset):
    """Rows: header path,label[,method,domain,identity]. Returns (img, label)."""
    def __init__(self, csv_path, transform=None):
        self.rows = []
        with open(csv_path, newline="") as f:
            for r in csv.DictReader(f):
                self.rows.append((r["path"], int(r["label"])))
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        path, label = self.rows[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

def read_csv_rows(csv_path):
    with open(csv_path, newline="") as f:
        return list(csv.DictReader(f))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"Device={device}")
print(f"Checkpoint exists: {CHECKPOINT.exists()}  ({CHECKPOINT.stat().st_size/1e6:.0f} MB)" if CHECKPOINT.exists()
      else f"Checkpoint MISSING: {CHECKPOINT}")
print(f"Manifest exists: {MANIFEST.exists()}")
print(f"Output dir: {OUT}")


PROJECT_ROOT=/workspace/quangmanh/deepfake
Device=cuda
Checkpoint exists: True  (116 MB)
Manifest exists: True
Output dir: /workspace/quangmanh/deepfake/experiments/results/error_analysis_lora


## Step 2 — Build a balanced test set from the manifest

`test_data_v3/manifest.csv` has 30,691 rows (1,177 real / 29,514 fake). We keep **all 1,177 real** frames and
draw **1,177 fake** frames **proportionally across methods**, so the test stays balanced *and* preserves the
method distribution of the fake population. The result carries `path, label, method, domain, identity` so all
later error analysis is metadata-driven (no fragile path parsing).

In [12]:
# ============================================================
# Step 2: Build balanced test set from the DF40 manifest
# ============================================================
def proportional_sample_by_method(fake_rows, n, seed):
    """Draw n rows across methods proportionally to method size (largest-remainder rounding)."""
    rng = random.Random(seed)
    by_method = {}
    for r in fake_rows:
        by_method.setdefault(r["method"], []).append(r)
    counts = {m: int(round(n * len(lst) / len(fake_rows))) for m, lst in by_method.items()}
    for m, lst in by_method.items():
        counts[m] = min(counts[m], len(lst))
    remaining = n - sum(counts.values())
    methods = sorted(by_method)
    i = 0
    while remaining != 0 and i < 100000:
        m = methods[i % len(methods)]
        lst = by_method[m]
        if remaining > 0 and counts[m] < len(lst):
            counts[m] += 1; remaining -= 1
        elif remaining < 0 and counts[m] > 0:
            counts[m] -= 1; remaining += 1
        i += 1
    sel = []
    for m, lst in by_method.items():
        sel.extend(rng.sample(lst, counts[m]))
    return sel

rows = read_csv_rows(MANIFEST)
real = [r for r in rows if r["label"] == "0"]
fake = [r for r in rows if r["label"] == "1"]
N_TEST_FAKE = len(real)                      # match real count -> balanced
sel_fake = proportional_sample_by_method(fake, N_TEST_FAKE, SEED)

test_rows = []
for r in real:
    test_rows.append((str(TEST_ROOT / r["path"]), 0, r["method"], r["domain"], r["identity"]))
for r in sel_fake:
    test_rows.append((str(TEST_ROOT / r["path"]), 1, r["method"], r["domain"], r["identity"]))
random.Random(SEED).shuffle(test_rows)

missing = [p for p, *_ in test_rows if not Path(p).exists()]
assert not missing, f"{len(missing)} test images missing, e.g. {missing[0]}"

with open(OUT / "test_balanced.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "method", "domain", "identity"])
    w.writerows(test_rows)

n_real = sum(1 for r in test_rows if r[1] == 0)
n_fake = sum(1 for r in test_rows if r[1] == 1)
print(f"test_balanced.csv: {len(test_rows)} rows ({n_real} real / {n_fake} fake)")
print(f"  fake methods covered: {len({r[2] for r in test_rows if r[1]==1})}")
print(f"  -> {OUT / 'test_balanced.csv'}")


test_balanced.csv: 2354 rows (1177 real / 1177 fake)
  fake methods covered: 40
  -> /workspace/quangmanh/deepfake/experiments/results/error_analysis_lora/test_balanced.csv


## Step 3 — Load the LoRA classifier and run predictions

The checkpoint is **self-contained**: full backbone (frozen pretrained ViT-S/16 Plus) + LoRA adapters on
`q_proj`/`v_proj` + the linear head. We rebuild the exact architecture (`DinoViT` → `apply_lora` →
`BackboneClassifier`), load the `state_dict`, and predict on the balanced test set. Predictions are cached to
`predictions.npz` so re-running skips the forward pass.

In [13]:
# ============================================================
# Step 3: Load LoRA classifier + predict on test set
# (fast; predictions cached to predictions.npz)
# ============================================================
from src.models.dinov3_vit import DinoViT
from src.models.lora import apply_lora

class BackboneClassifier(nn.Module):
    """Matches src/training/finetune_lora.py — backbone (with LoRA) + Linear head."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.embed_dim, 2)
    def forward(self, x):
        return self.head(self.backbone(x))

predictions_path = OUT / "predictions.npz"
if predictions_path.exists():
    d = np.load(predictions_path)
    preds, probs, labels = d["preds"], d["probs"], d["labels"]
    print("Loaded cached predictions.")
else:
    ck = torch.load(str(CHECKPOINT), map_location="cpu", weights_only=True)
    lc = ck["lora_config"]
    print(f"Checkpoint: epoch={ck['epoch']} | lora_config={lc}")
    print(f"  saved val_metrics: {ck['val_metrics']}")

    # ViT-S/16 Plus architecture (matches dinov3-vits16plus-pretrain-lvd1689m)
    backbone = DinoViT(img_size=IMG_SIZE, embed_dim=384, depth=12, num_heads=6,
                       mlp_ratio=4.0, num_registers=4, gated_mlp=True)
    n_wrap = apply_lora(backbone, r=lc["r"], alpha=lc["alpha"], targets=lc["targets"])
    model = BackboneClassifier(backbone)
    model.load_state_dict(ck["state_dict"])          # self-contained: backbone + LoRA + head
    model.eval().to(device)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"Model loaded: {n_all/1e6:.1f}M params, LoRA wrapped {n_wrap} projections (q/v). Predicting...")

    ds = CsvImageDataset(OUT / "test_balanced.csv", transform=EVAL_TF)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Predicting"):
            x = x.to(device)
            logits = model(x)
            p = torch.softmax(logits, dim=1)
            preds.extend(logits.argmax(1).cpu().tolist())
            probs.extend(p[:, 1].cpu().tolist())
            labels.extend(y.tolist())
    preds = np.array(preds); probs = np.array(probs); labels = np.array(labels)
    np.savez_compressed(predictions_path, preds=preds, probs=probs, labels=labels)
    print(f"Saved -> {predictions_path}")

print(f"Test samples: {len(labels)} | real: {(labels==0).sum()} | fake: {(labels==1).sum()}")


Loaded cached predictions.
Test samples: 2354 | real: 1177 | fake: 1177


## Step 4 — Confusion matrix & error masks

Load predictions and the per-sample metadata, then isolate the False Negatives (fakes missed) and
False Positives (reals flagged).

In [14]:
# ============================================================
# Step 4: Confusion matrix + error masks (CPU)
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

d = np.load(OUT / "predictions.npz")
preds, probs, labels = d["preds"], d["probs"], d["labels"]
meta = read_csv_rows(OUT / "test_balanced.csv")
paths      = np.array([r["path"] for r in meta])
methods    = np.array([r["method"] for r in meta])
domains    = np.array([r["domain"] for r in meta])
identities = np.array([r["identity"] for r in meta])
assert len(meta) == len(labels), "metadata / predictions length mismatch"

cm = confusion_matrix(labels, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}  (total={tn+fp+fn+tp})")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real (0)", "Fake (1)"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — LoRA fine-tuned DINOv3 ViT-S/16 Plus (test_balanced)")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=150)
plt.show()

fn_mask = (labels == 1) & (preds == 0)   # fakes predicted as real
fp_mask = (labels == 0) & (preds == 1)   # reals predicted as fake
print(f"Error counts -> FN(fake missed)={fn_mask.sum()}, FP(real flagged)={fp_mask.sum()}")


TN=1125 FP=52 FN=75 TP=1102  (total=2354)
Error counts -> FN(fake missed)=75, FP(real flagged)=52


## Step 5 — Error categorization by method and domain

Which deepfake methods / image domains are driving the errors? Tables are computed with `numpy` grouping.

In [15]:
# ============================================================
# Step 5: Error rate by domain and by method
# ============================================================
print("=== Error rate by DOMAIN ===")
print(f"{'domain':<8} {'n':>6} {'err':>5} {'FN':>5} {'FP':>5} {'err_rate':>9}")
for dom in sorted(set(domains)):
    m = domains == dom
    n = int(m.sum()); err = int((labels[m] != preds[m]).sum())
    fn_ = int(((labels[m] == 1) & (preds[m] == 0)).sum())
    fp_ = int(((labels[m] == 0) & (preds[m] == 1)).sum())
    print(f"{dom:<8} {n:>6} {err:>5} {fn_:>5} {fp_:>5} {err/n:>9.4f}")

print("\n=== Per-method FN rate (fake methods only, worst first) ===")
rows_m = []
for mth in sorted(set(methods)):
    if mth == "real":
        continue
    m = methods == mth
    n = int(m.sum())
    fn_ = int(((labels[m] == 1) & (preds[m] == 0)).sum())
    det = float((preds[m] == 1).mean())
    rows_m.append((mth, n, fn_, det))
rows_m.sort(key=lambda t: t[3])          # worst detection first
print(f"{'method':<16} {'n':>5} {'FN':>4} {'det_rate':>9}")
for mth, n, fn_, det in rows_m:
    print(f"{mth:<16} {n:>5} {fn_:>4} {det:>9.4f}")

rows_m.sort(key=lambda t: t[3])
top = rows_m[:15]
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh([t[0] for t in top], [1 - t[3] for t in top])
ax.invert_yaxis()
ax.set_xlabel("FN rate (missed fakes / method samples)")
ax.set_title("False-Negative Rate by Deepfake Method — LoRA ViT-S/16 Plus")
plt.tight_layout()
plt.savefig(OUT / "fn_by_method.png", dpi=150)
plt.show()


=== Error rate by DOMAIN ===
domain        n   err    FN    FP  err_rate
cdc         251     5     5     0    0.0199
efs         242    28    28     0    0.1157
fe          120     7     7     0    0.0583
ffc        1242    71    19    52    0.0572
oth         499    16    16     0    0.0321

=== Per-method FN rate (fake methods only, worst first) ===
method               n   FN  det_rate
MidJourney          26   17    0.3462
facedancer          27    9    0.6667
whichfaceisreal     30    8    0.7333
faceswap            27    6    0.7778
inswap              19    4    0.7895
styleclip           40    7    0.8250
fsgan               26    4    0.8462
sadtalker           26    4    0.8462
MRAA                29    3    0.8966
CollabDiff          31    3    0.9032
wav2lip             22    2    0.9091
pirender            27    2    0.9259
e4s                 15    1    0.9333
mobileswap          56    3    0.9464
fomm                27    1    0.9630
pixart              37    1    0.9730


## Step 6 — Qualitative inspection of misclassified images

Grid views of the errors let you judge *visually* why the model failed — compression artifacts, lighting,
method artifacts, or genuinely hard samples.

In [16]:
# ============================================================
# Step 6: Error image galleries (FN / FP / worst methods)
# ============================================================
def make_error_grid(mask, title, n=12, ncols=4):
    idx = np.where(mask)[0]
    if len(idx) == 0:
        print(f"No samples for '{title}'"); return None
    sel = idx[np.argsort(np.abs(probs[idx] - 0.5))][:n] if len(idx) > n else idx
    nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.2))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for ax, i in zip(axes, sel):
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(np.array(img))
        ax.set_title(f"{methods[i]} | p={probs[i]:.2f}", fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    return fig

fig = make_error_grid(fn_mask, "False Negatives — fakes the model called REAL (n=%d)" % fn_mask.sum())
if fig:
    fig.savefig(OUT / "errors_fn.png", dpi=140); plt.show()

fig = make_error_grid(fp_mask, "False Positives — reals the model called FAKE (n=%d)" % fp_mask.sum())
if fig:
    fig.savefig(OUT / "errors_fp.png", dpi=140); plt.show()

# ---- grid ALL missed fakes of the worst methods ----
def grid_all_method(method, ncols=4, max_cells=60):
    sub_idx = np.where((methods == method) & (labels == 1) & (preds == 0))[0]
    if len(sub_idx) == 0:
        print(f"[{method}] no FN samples"); return None
    sel = sub_idx[:max_cells]
    ncols = min(ncols, len(sel)); nrows = int(np.ceil(len(sel) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.0, nrows * 3.0))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for ax, i in zip(axes, sel):
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(np.array(img))
        ax.set_title(f"p={probs[i]:.2f}\n{identities[i]}", fontsize=8)
    fig.suptitle(f"{method} — missed fakes (n_FN={len(sub_idx)})", fontsize=12)
    plt.tight_layout()
    return fig

fn_rate = {}
for mth in set(methods):
    if mth == "real":
        continue
    m = (methods == mth) & (labels == 1)
    if m.sum() > 0:
        fn_rate[mth] = ((preds[m] == 0).sum()) / m.sum()
worst = sorted(fn_rate, key=fn_rate.get, reverse=True)[:4]
print("Worst methods (highest FN rate):", {m: round(fn_rate[m], 3) for m in worst})

for method in worst:
    fig = grid_all_method(method)
    if fig is not None:
        safe = method.replace("/", "_")
        fig.savefig(OUT / f"worst_method_{safe}.png", dpi=130); plt.show()


Worst methods (highest FN rate): {np.str_('MidJourney'): np.float64(0.654), np.str_('facedancer'): np.float64(0.333), np.str_('whichfaceisreal'): np.float64(0.267), np.str_('faceswap'): np.float64(0.222)}


KeyboardInterrupt: 

## Step 7 — Error distribution over image statistics

Brightness / std-dev are cheap proxies for lighting, occlusion, compression and background noise.
Compare correct vs error subsets to see whether errors cluster on dark or low-texture images.

In [ ]:
# ============================================================
# Step 7: Image statistics — correct vs errors
# ============================================================
def img_stats(paths_subset):
    bright, stdv = [], []
    for p in tqdm(paths_subset, desc="Computing image stats"):
        g = np.array(Image.open(p).convert("L")).astype(np.float32)
        bright.append(g.mean()); stdv.append(g.std())
    return np.array(bright), np.array(stdv)

ok_mask = labels == preds
err_mask = labels != preds
rng = np.random.RandomState(0)
idx_ok = rng.choice(np.where(ok_mask)[0], min(1500, ok_mask.sum()), replace=False)
idx_err = np.where(err_mask)[0]

b_ok, s_ok = img_stats(paths[idx_ok])
b_err, s_err = img_stats(paths[idx_err])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(b_ok, bins=40, alpha=0.6, label="Correct")
axes[0].hist(b_err, bins=40, alpha=0.6, label="Errors")
axes[0].set_xlabel("Mean brightness"); axes[0].set_title("Brightness: Correct vs Errors"); axes[0].legend()
axes[1].hist(s_ok, bins=40, alpha=0.6, label="Correct")
axes[1].hist(s_err, bins=40, alpha=0.6, label="Errors")
axes[1].set_xlabel("Std-dev (contrast/texture)"); axes[1].set_title("Std-dev: Correct vs Errors"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT / "error_image_statistics.png", dpi=140)
plt.show()

print(f"Brightness: correct {b_ok.mean():.1f}+-{b_ok.std():.1f} | errors {b_err.mean():.1f}+-{b_err.std():.1f}")
print(f"Std-dev:    correct {s_ok.mean():.1f}+-{s_ok.std():.1f} | errors {s_err.mean():.1f}+-{s_err.std():.1f}")


## Step 8 — Export summary

Write the aggregate metrics to JSON for later comparison against other backbones / training runs.

In [ ]:
# ============================================================
# Step 8: Export summary
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

summary = {
    "model": "dinov3-vits16plus-pretrain-lvd1689m + LoRA (r=16, alpha=32, q/v)",
    "ckpt": str(CHECKPOINT),
    "n_test": int(len(labels)),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    "accuracy": float((labels == preds).mean()),
    "precision": float(precision_score(labels, preds, zero_division=0)),
    "recall": float(recall_score(labels, preds, zero_division=0)),
    "f1": float(f1_score(labels, preds, zero_division=0)),
    "roc_auc": float(roc_auc_score(labels, probs)),
}
with open(OUT / "error_analysis_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("\nArtifacts written to:", OUT)
print("  - test_balanced.csv, predictions.npz, error_analysis_summary.json")
print("  - confusion_matrix.png, fn_by_method.png, errors_fn.png, errors_fp.png, error_image_statistics.png")
